# 🧩 Mini-Lab: Hosted Backend (Fixture Path Required)

**Module 6: The Replaceable Model Interface** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Call** a hosted AI API using a base URL, an environment-variable key, and an explicit timeout
2. **Apply** secrets hygiene by loading API keys from environment variables instead of hard-coding them
3. **Implement** a hosted backend that conforms to `AIClient` and **runs against recorded fixtures when no key is present**
4. **Parse** API responses defensively

## Target Concepts

| Concept | Gloss |
|---------|-------|
| HTTP API Request | Base URL, env-var key, explicit timeout, status handling |
| Secrets Hygiene | Keep keys out of code, logs, and git |
| Interface Seam | Adapter returns your dataclass, never the vendor object |
| Output Parsing | Model output may not be valid JSON; parse and guard |


## Recap

In **mini-define-ai-protocol**, we defined an `AIClient` protocol with `complete(AIRequest) -> AIResponse` and built a mock backend. In **mini-typed-prompt-parser**, we learned to parse LLM responses defensively.

Now we'll build a **real hosted backend** — one that actually calls an API over HTTP. This is "Backend 2" in our module: a class that speaks HTTP to a remote service but still satisfies the same `AIClient` protocol. The business logic won't know the difference.

## Step 1 — Shared Types from Previous Labs

We reuse `AIRequest`, `AIResponse`, and the `AIClient` protocol (same as **mini-define-ai-protocol**).

In [2]:
from dataclasses import dataclass, field
from typing import Protocol, runtime_checkable, Optional


@dataclass
class AIRequest:
    """What the business logic sends to the AI backend."""
    prompt: str
    system: str = "You are a helpful assistant."
    temperature: float = 0.0
    max_tokens: int = 256


@dataclass
class AIResponse:
    """What the AI backend returns to the business logic."""
    text: str
    model: str = "unknown"
    usage: dict = field(default_factory=dict)


@runtime_checkable
class AIClient(Protocol):
    """Any AI backend must implement this single method."""
    def complete(self, request: AIRequest) -> AIResponse: ...


print("Shared types ready: AIRequest, AIResponse, AIClient protocol.")

Shared types ready: AIRequest, AIResponse, AIClient protocol.


## Step 2 — Secrets Hygiene: Loading the API Key

Load the key from an environment variable, never hard-code it, and never print the full key.

In [3]:
import os
from dotenv import load_dotenv

# Load .env file (if it exists) into environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    # Show only the first 5 and last 3 characters — never the full key
    masked = api_key[:5] + "..." + api_key[-3:]
    print(f"✅ API key loaded (masked): {masked}")
else:
    print("⚠️  OPENAI_API_KEY missing — the fixture path will be used.")

✅ API key loaded (masked): sk-pr...KoA


## Step 3 — The Hosted Backend: Anatomy of an HTTP API Call

> 🧩 **HTTP API request.** The call pattern from the video, now wrapped in a backend.

A hosted AI API call has four ingredients:

1. **Base URL** — the endpoint (`https://api.openai.com/v1`)
2. **Auth header** — `Authorization: Bearer <key>`
3. **Request body** — JSON payload with model, messages, parameters
4. **Timeout** — how long to wait before giving up

We wrap all four behind the `AIClient` protocol.

In [4]:
import socket

import requests
from urllib3.util import connection as urllib3_connection

# Windows often resets the TLS handshake to this host over IPv6 (WinError 10054).
urllib3_connection.allowed_gai_family = lambda: socket.AF_INET


class HostedAIClient:
    """AI backend that calls a hosted API over HTTP."""

    def __init__(
        self,
        api_key: str,
        base_url: str = "https://api.openai.com/v1",
        model: str = "gpt-4o-mini",
        timeout: int = 30,
    ):
        # The caller loads the key from the environment and passes it in
        self.api_key = api_key
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = timeout

    def complete(self, request: AIRequest) -> AIResponse:
        """Send a request to the hosted API and return an AIResponse."""
        url = f"{self.base_url}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": request.system},
                {"role": "user", "content": request.prompt},
            ],
            "temperature": request.temperature,
            "max_tokens": request.max_tokens,
        }

        # The actual HTTP call — always with an explicit timeout
        resp = requests.post(url, headers=headers, json=payload, timeout=self.timeout)
        resp.raise_for_status()  # turns 4xx/5xx into an exception

        # Defensive parsing: .get() chains give empty values instead of a KeyError
        body = resp.json()
        text = body.get("choices", [{}])[0].get("message", {}).get("content", "")
        return AIResponse(
            text=text.strip(),
            model=body.get("model", self.model),
            usage=body.get("usage", {}),
        )


print("HostedAIClient defined.")

HostedAIClient defined.


## Step 4 — Recorded fixtures first, live key optional

The required path uses recorded fixtures when `OPENAI_API_KEY` is missing. They have the same `AIResponse` shape, so the pipeline still runs. A live call is optional.

In [5]:
# Recorded fixtures — same AIResponse shape as a hosted reply, no network needed.
FIXTURES = {
    "billing": AIResponse(text="billing", model="gpt-4o-mini-fixture", usage={"prompt_tokens": 24, "completion_tokens": 1}),
    "default": AIResponse(text="other", model="gpt-4o-mini-fixture", usage={"prompt_tokens": 20, "completion_tokens": 1}),
}


def complete_with_fallback(client: HostedAIClient, request: AIRequest) -> AIResponse:
    """Call the hosted API when a key is set; otherwise return a recorded fixture."""
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if key:
        print("Using hosted API.")
        return client.complete(request)

    print("No OPENAI_API_KEY — using recorded fixture.")
    if "charged twice" in request.prompt.lower():
        return FIXTURES["billing"]
    return FIXTURES["default"]


hosted = HostedAIClient(
    api_key=os.getenv("OPENAI_API_KEY", "sk-fixture"),
    model="gpt-4o-mini",
    timeout=15,
)

request = AIRequest(
    prompt="Classify this support ticket: 'I was charged twice this month'. Category:",
    system="You are a support classifier. Reply with exactly one word: billing, technical, account, or other.",
    temperature=0.0,
    max_tokens=10,
)

response = complete_with_fallback(hosted, request)
print(f"Category: {response.text}  (model: {response.model})")

Using hosted API.
Category: billing  (model: gpt-4o-mini-2024-07-18)


In [6]:
print(f"Response text: {response.text}\nModel: {response.model}\nTokens: {response.usage}")

Response text: billing
Model: gpt-4o-mini-2024-07-18
Tokens: {'prompt_tokens': 48, 'completion_tokens': 1, 'total_tokens': 49, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}}


## Step 5 — The Interface Seam in Action

Now run `classify_ticket` (same as **mini-define-ai-protocol**) with the hosted-or-fixture backend to prove the seam.

In [7]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))


def classify_ticket(ticket_text: str, ai_client: AIClient) -> str:
    """Classify a support ticket. The AI backend is injected."""
    categories = ["billing", "technical", "account", "other"]
    request = AIRequest(
        prompt=(
            f"Classify this support ticket into one of {categories}.\n"
            f"Ticket: {ticket_text}\nCategory:"
        ),
        system="Reply with exactly one category name, nothing else.",
        temperature=0.0,
        max_tokens=10,
    )
    response = ai_client.complete(request)
    category = response.text.strip().lower()
    if category not in categories:
        category = "other"
    return category


# Run with the hosted backend
ticket = "My login keeps failing with error 403"
result = classify_ticket(ticket, hosted)

md(f"""
| Detail | Value |
|--------|-------|
| **Ticket** | {ticket} |
| **Backend** | `HostedAIClient` → `{hosted.model}` |
| **Timeout** | {hosted.timeout}s |
| **Result** | `{result}` |

The function `classify_ticket` has **zero vendor imports** — it works with
the mock or the hosted backend through the same `AIClient` protocol.
""")


| Detail | Value |
|--------|-------|
| **Ticket** | My login keeps failing with error 403 |
| **Backend** | `HostedAIClient` → `gpt-4o-mini` |
| **Timeout** | 15s |
| **Result** | `technical` |

The function `classify_ticket` has **zero vendor imports** — it works with
the mock or the hosted backend through the same `AIClient` protocol.


## 🎯 Summary

### Key Takeaways

1. **HTTP API call anatomy** — a hosted AI call needs four things: a base URL, an auth header with the API key, a JSON payload, and an explicit timeout
2. **Secrets hygiene** — load API keys from environment variables (`os.getenv`), never hard-code them, and mask before printing; the backend class receives the key as a parameter, never reads the environment itself
3. **Explicit timeout** — without a `timeout` parameter, HTTP requests can hang indefinitely; always set one to keep failures recoverable
4. **Interface seam preserved** — `HostedAIClient` satisfies the same `AIClient` protocol as the mock backend, so business logic stays vendor-free and testable